# M-PESA Transaction Anomaly Analysis
Covers: dataset overview · fraud patterns · temporal trends · amount distributions · user behaviour · anomaly indicators

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('transactions.csv', parse_dates=['timestamp'])
print(f'Loaded {len(df):,} rows')
df.head()

## 1. Dataset Overview

In [ ]:
print(f"Shape          : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Date range     : {df['timestamp'].min().date()} → {df['timestamp'].max().date()}")
print(f"Unique users   : {df['user_id'].nunique():,}")
print(f"Transaction types: {sorted(df['transaction_type'].unique())}")
print()
print('Missing values:')
print(df.isnull().sum())

In [ ]:
df.describe().round(2)

## 2. Fraud Distribution

In [ ]:
fraud_counts = df['is_fraud'].value_counts()
fraud_rate   = df['is_fraud'].mean() * 100

print(f"Legitimate : {fraud_counts[0]:,}  ({100 - fraud_rate:.2f}%)")
print(f"Fraudulent : {fraud_counts[1]:,}  ({fraud_rate:.2f}%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar
fraud_counts.plot.bar(ax=axes[0], color=['steelblue', 'crimson'], edgecolor='white')
axes[0].set_xticklabels(['Legitimate', 'Fraudulent'], rotation=0)
axes[0].set_title('Transaction Count by Fraud Label')
axes[0].set_ylabel('Count')
for p in axes[0].patches:
    axes[0].annotate(f"{int(p.get_height()):,}",
                     (p.get_x() + p.get_width() / 2, p.get_height()),
                     ha='center', va='bottom', fontsize=10)

# Pie
axes[1].pie(fraud_counts, labels=['Legitimate', 'Fraudulent'],
            autopct='%1.2f%%', colors=['steelblue', 'crimson'], startangle=90)
axes[1].set_title('Fraud Share')
plt.tight_layout()
plt.show()

## 3. Fraud Rate by Transaction Type

In [ ]:
fraud_by_type = (
    df.groupby('transaction_type')['is_fraud']
    .agg(total='count', fraud_count='sum')
    .assign(fraud_rate=lambda x: (x['fraud_count'] / x['total'] * 100).round(2))
    .sort_values('fraud_rate', ascending=False)
)
display(fraud_by_type)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

fraud_by_type['fraud_rate'].plot.barh(ax=axes[0], color='coral', edgecolor='white')
axes[0].set_title('Fraud Rate by Transaction Type (%)')
axes[0].set_xlabel('Fraud Rate (%)')

fraud_by_type['total'].plot.barh(ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Transaction Volume by Type')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

## 4. Temporal Patterns

In [ ]:
day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

hourly_fraud  = df.groupby('hour_of_day')['is_fraud'].mean() * 100
hourly_volume = df.groupby('hour_of_day')['transaction_id'].count()
daily_fraud   = df.groupby('day_of_week')['is_fraud'].mean() * 100

fig, axes = plt.subplots(2, 2, figsize=(16, 8))

# Hourly fraud rate
axes[0, 0].plot(hourly_fraud.index, hourly_fraud.values, marker='o', color='darkorange', linewidth=2)
axes[0, 0].set_title('Fraud Rate by Hour of Day')
axes[0, 0].set_xlabel('Hour')
axes[0, 0].set_ylabel('Fraud Rate (%)')
axes[0, 0].set_xticks(range(0, 24))

# Hourly transaction volume
axes[0, 1].bar(hourly_volume.index, hourly_volume.values, color='steelblue', edgecolor='white')
axes[0, 1].set_title('Transaction Volume by Hour of Day')
axes[0, 1].set_xlabel('Hour')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_xticks(range(0, 24))

# Fraud rate by day of week
axes[1, 0].bar([day_labels[i] for i in daily_fraud.index], daily_fraud.values,
               color='coral', edgecolor='white')
axes[1, 0].set_title('Fraud Rate by Day of Week')
axes[1, 0].set_ylabel('Fraud Rate (%)')

# Heatmap: fraud rate by hour × transaction type
pivot = df.pivot_table(values='is_fraud', index='transaction_type',
                        columns='hour_of_day', aggfunc='mean') * 100
sns.heatmap(pivot, ax=axes[1, 1], cmap='YlOrRd', linewidths=0.3,
            cbar_kws={'label': 'Fraud Rate (%)'})
axes[1, 1].set_title('Fraud Rate: Transaction Type × Hour')
axes[1, 1].set_xlabel('Hour of Day')

plt.tight_layout()
plt.show()

## 5. Binary Flag Analysis (Night / Weekend / Large)

In [ ]:
flag_cols = ['is_night', 'is_weekend', 'is_large_tx']

flag_rates = pd.DataFrame({
    col: {
        'overall_rate_%': df[col].mean() * 100,
        'fraud_rate_when_0_%': df[df[col] == 0]['is_fraud'].mean() * 100,
        'fraud_rate_when_1_%': df[df[col] == 1]['is_fraud'].mean() * 100,
    }
    for col in flag_cols
}).T.round(2)

display(flag_rates)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, col in zip(axes, flag_cols):
    rates = df.groupby(col)['is_fraud'].mean() * 100
    ax.bar(['No', 'Yes'], [rates.get(0, 0), rates.get(1, 0)],
           color=['steelblue', 'crimson'], edgecolor='white')
    ax.set_title(f'Fraud Rate: {col}')
    ax.set_ylabel('Fraud Rate (%)')
    for i, v in enumerate([rates.get(0, 0), rates.get(1, 0)]):
        ax.text(i, v + 0.05, f'{v:.2f}%', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

## 6. Amount Analysis

In [ ]:
amount_summary = df.groupby('is_fraud')['amount_kes'].describe().round(2)
amount_summary.index = ['Legitimate', 'Fraudulent']
display(amount_summary)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Histogram (log scale)
for label, mask, color in [('Legitimate', df['is_fraud'] == 0, 'steelblue'),
                             ('Fraudulent', df['is_fraud'] == 1, 'crimson')]:
    axes[0].hist(np.log1p(df.loc[mask, 'amount_kes']), bins=50,
                 alpha=0.6, label=label, color=color)
axes[0].set_title('Amount Distribution (log scale)')
axes[0].set_xlabel('log(1 + Amount KES)')
axes[0].legend()

# Box plot
clipped = df[df['amount_kes'] < df['amount_kes'].quantile(0.99)]
sns.boxplot(data=clipped, x='is_fraud', y='amount_kes', ax=axes[1],
            palette={0: 'steelblue', 1: 'crimson'})
axes[1].set_xticklabels(['Legitimate', 'Fraudulent'])
axes[1].set_title('Amount by Fraud Label (p99 cap)')
axes[1].set_ylabel('Amount (KES)')

# Mean amount by transaction type
mean_amt = df.groupby('transaction_type')['amount_kes'].mean().sort_values()
mean_amt.plot.barh(ax=axes[2], color='teal', edgecolor='white')
axes[2].set_title('Mean Amount by Transaction Type')
axes[2].set_xlabel('Mean Amount (KES)')

plt.tight_layout()
plt.show()

## 7. Amount Deviation Analysis

In [ ]:
dev_summary = df.groupby('is_fraud')['amount_deviation'].describe().round(2)
dev_summary.index = ['Legitimate', 'Fraudulent']
display(dev_summary)

thresholds = [2, 5, 10, 20, 50]
print('\nFraud rate above deviation threshold:')
for t in thresholds:
    subset = df[df['amount_deviation'] > t]
    rate   = subset['is_fraud'].mean() * 100 if len(subset) else 0
    print(f'  deviation > {t:3d}x : {len(subset):5,} tx  →  fraud rate {rate:.2f}%')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

clipped_dev = df[df['amount_deviation'] < df['amount_deviation'].quantile(0.99)]
sns.boxplot(data=clipped_dev, x='is_fraud', y='amount_deviation', ax=axes[0],
            palette={0: 'steelblue', 1: 'crimson'})
axes[0].set_xticklabels(['Legitimate', 'Fraudulent'])
axes[0].set_title('Amount Deviation by Fraud Label (p99 cap)')
axes[0].set_ylabel('Deviation (× user avg)')

# Scatter: deviation vs amount, coloured by fraud
sample = df.sample(min(3000, len(df)), random_state=42)
axes[1].scatter(np.log1p(sample['amount_kes']), sample['amount_deviation'],
                c=sample['is_fraud'].map({0: 'steelblue', 1: 'crimson'}),
                alpha=0.4, s=15)
axes[1].set_title('Amount vs Deviation (blue=legit, red=fraud)')
axes[1].set_xlabel('log(1 + Amount KES)')
axes[1].set_ylabel('Deviation (× user avg)')

plt.tight_layout()
plt.show()

## 8. User Behaviour

In [ ]:
user_stats = df.groupby('user_id').agg(
    tx_count=('transaction_id', 'count'),
    fraud_count=('is_fraud', 'sum'),
    avg_amount=('amount_kes', 'mean'),
    max_deviation=('amount_deviation', 'max'),
)
user_stats['fraud_rate_%'] = (user_stats['fraud_count'] / user_stats['tx_count'] * 100).round(2)

print(f"Avg tx per user     : {user_stats['tx_count'].mean():.1f}")
print(f"Max tx per user     : {user_stats['tx_count'].max()}")
print(f"Users with ≥1 fraud : {(user_stats['fraud_count'] > 0).sum()}")
print()
print('Top 10 users by fraud count:')
display(user_stats.nlargest(10, 'fraud_count').round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(user_stats['tx_count'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Transactions per User')
axes[0].set_xlabel('Transaction Count')
axes[0].set_ylabel('Number of Users')

top_users = user_stats.nlargest(15, 'fraud_count').reset_index()
axes[1].barh(top_users['user_id'], top_users['fraud_count'],
             color='crimson', edgecolor='white')
axes[1].set_title('Top 15 Users by Fraud Count')
axes[1].set_xlabel('Fraud Transactions')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 9. Time Between Transactions

In [ ]:
td = df['time_diff'].dropna()
print(f"Rows with time_diff : {len(td):,}  (missing: {df['time_diff'].isna().sum():,})")
print(f"Median gap          : {td.median() / 3600:.1f} hours")
print(f"Mean gap            : {td.mean() / 3600:.1f} hours")
print(f"Rapid tx (<60 s)    : {(td < 60).sum():,}")

rapid_fraud = df[df['time_diff'] < 60]['is_fraud'].mean() * 100
print(f"Fraud rate (<60 s)  : {rapid_fraud:.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(np.log1p(td), bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Time Between Transactions (log scale)')
axes[0].set_xlabel('log(1 + seconds)')
axes[0].set_ylabel('Count')

td_by_fraud = df.dropna(subset=['time_diff'])
for label, mask, color in [('Legitimate', td_by_fraud['is_fraud'] == 0, 'steelblue'),
                             ('Fraudulent', td_by_fraud['is_fraud'] == 1, 'crimson')]:
    axes[1].hist(np.log1p(td_by_fraud.loc[mask, 'time_diff']),
                 bins=60, alpha=0.6, label=label, color=color)
axes[1].set_title('Time Between Tx by Fraud Label')
axes[1].set_xlabel('log(1 + seconds)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 10. Correlation & Feature Relationships

In [ ]:
numeric_cols = ['amount_kes', 'hour_of_day', 'day_of_week', 'day_of_month',
                'user_avg_amount', 'amount_deviation', 'time_diff',
                'is_night', 'is_large_tx', 'is_weekend', 'is_fraud']

corr = df[numeric_cols].corr()

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Full correlation heatmap
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, ax=axes[0], cmap='coolwarm', center=0,
            annot=True, fmt='.2f', linewidths=0.5, square=True)
axes[0].set_title('Feature Correlation Matrix')

# Correlation with is_fraud only
fraud_corr = corr['is_fraud'].drop('is_fraud').sort_values()
colors = ['crimson' if v > 0 else 'steelblue' for v in fraud_corr]
fraud_corr.plot.barh(ax=axes[1], color=colors, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Feature Correlation with is_fraud')
axes[1].set_xlabel('Pearson r')

plt.tight_layout()
plt.show()

## 11. Key Findings Summary

In [ ]:
night_fraud  = df[df['is_night']   == 1]['is_fraud'].mean() * 100
large_fraud  = df[df['is_large_tx']== 1]['is_fraud'].mean() * 100
high_dev     = df[df['amount_deviation'] > 10]['is_fraud'].mean() * 100
rapid        = df[df['time_diff']  < 60]['is_fraud'].mean() * 100
top_type     = fraud_by_type['fraud_rate'].idxmax()
top_type_pct = fraud_by_type['fraud_rate'].max()

print('=' * 55)
print('KEY FINDINGS')
print('=' * 55)
print(f'  Overall fraud rate          : {fraud_rate:.2f}%')
print(f'  Fraud rate at night         : {night_fraud:.2f}%')
print(f'  Fraud rate for large tx     : {large_fraud:.2f}%')
print(f'  Fraud rate (deviation >10x) : {high_dev:.2f}%')
print(f'  Fraud rate (rapid <60 s)    : {rapid:.2f}%')
print(f'  Highest-risk tx type        : {top_type} ({top_type_pct:.2f}%)')
print()
print('  Strong fraud predictors (from correlation):')
top_predictors = fraud_corr.abs().sort_values(ascending=False).head(5)
for feat, r in top_predictors.items():
    direction = '+' if fraud_corr[feat] > 0 else '-'
    print(f'    {direction} {feat:<20s}  r = {fraud_corr[feat]:.3f}')